# Mental Health in Tech Survey

The data used herein was sourced from:<br>
https://www.kaggle.com/datasets/osmi/mental-health-in-tech-survey/data

The `mhit` project is composed of x notebooks:<br>**A. Data transformations (THIS NOTEBOOK)**<br> 
Because the data is unwieldy in its original form, before it was used for analysis and insights, I transformed it as follows:<br> 1. Created a unique id for each survey respondent (`rid`) and mapped all responses to this id;<br>2. Mapped all survey questions to their unique identifiers (`Q1`-`Q63`) and used the question identifiers as compact column headers.<br>3. Structured responses referencing conditions by first mapping all conditions to their own unique identifier (`C1` - `C51`) and then mapping each respondent to all of the conditions they listed as one they either currently have or one they have been diagnosed with.<br>4. Similarly, mapped all job positions to unique identifiers (`P1`-`P15`) and all respondents to each position they listed as part of their current responsibilities.<br>5. Finally, the responses are captured as text and while most represent ordinal or quantifiable information, further analysis required mapping of the text to numeric values *(e.g. "Yes" = 1 / "No" = 0 or "Yes, all" = 1, "Some of them" = 0.5, "No, none" = 0)*<br>
**B. EDA and additional data cleaning**<br>**C. Analysis and hypothesis testing**<br>**D. NLP and Sentiment Analysis**<br>

### Library imports, notebook settings, and data files

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import uuid

In [ ]:
pd.set_option('display.max_columns', None)
%config InlineBackend.figure_format = 'retina'

In [3]:
mhit = pd.read_csv('data/mental-heath-in-tech-2016_20161114.csv')
mhit.sample()

,Are you self-employed?,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health concerns and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",...,"If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?","If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?",What is your age?,What is your gender?,What country do you live in?,What US state or territory do you live in?,What country do you work in?,What US state or territory do you work in?,Which of the following best describes your work position?,Do you work remotely?
0,0,26-100,1.0,NaN,Not eligible for coverage / N/A,NaN,No,No,I don't know,Very easy,...,Not applicable to me,Not applicable to me,39,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Sometimes
1,0,6-25,1.0,NaN,No,Yes,Yes,Yes,Yes,Somewhat easy,...,Rarely,Sometimes,29,male,United States of America,Illinois,United States of America,Illinois,Back-end Developer|Front-end Developer,Never
2,0,6-25,1.0,NaN,No,NaN,No,No,I don't know,Neither easy nor difficult,...,Not applicable to me,Not applicable to me,38,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Always
3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Sometimes,Sometimes,43,male,United Kingdom,NaN,United Kingdom,NaN,Supervisor/Team Lead,Sometimes
4,0,6-25,0.0,1.0,Yes,Yes,No,No,No,Neither easy nor difficult,...,Sometimes,Sometimes,43,Female,United States of America,Illinois,United States of America,Illinois,Executive Leadership|Supervisor/Team Lead|Dev ...,Sometimes


In [5]:
mhit.shape

(1433, 63)

### Data transformations

Because the data is unwieldy in its original form, before it was used for analysis and insights, I transformed it as follows:<br> 1. Created a unique id for each survey respondent (`rid`) and mapped all responses to this id;<br>2. Mapped all survey questions to their unique identifiers (`Q1`-`Q63`) and used the question identifiers as compact column headers.<br>3. Structured responses referencing conditions by first mapping all conditions to their own unique identifier (`C1` - `C51`) and then mapping each respondent to all of the conditions they listed as one they either currently have or one they have been diagnosed with.<br>4. Similarly, mapped all job positions to unique identifiers (`P1`-`P15`) and all respondents to each position they listed as part of their current responsibilities.<br>5. Finally, the responses are captured as text and while most represent ordinal or quantifiable information, further analysis required mapping of the text to numeric values *(e.g. "Yes" = 1 / "No" = 0 or "Yes, all" = 1, "Some of them" = 0.5, "No, none" = 0)*

#### 1. response identifiers

In [ ]:
identifiers = [str(uuid.uuid4()) for i in mhit.index]

mhit['rid'] = identifiers
mhit.set_index('rid', inplace = True)

mhit.to_csv('all_raw.csv', index_label = 'rid')

#### 2. question identifiers

In [48]:
mhit_question_ids = {c: f"Q{i+1}" for i, c in enumerate(mhit.columns)}
mhit_question_ids_rev = {i: q for (q, i) in mhit_question_ids.items()}

In [1391]:
all_question_ids = pd.DataFrame.from_dict({q: [mhit_question_ids[q]] for q in mhit.columns}).T.reset_index().rename({'index': 'question',
                                                                                           0: 'qid'}, axis = 1)
all_question_ids.head(3)

In [571]:
all_question_ids.to_csv('all_question_ids.csv',  index = 0)

#### 3. conditions: unstructured --> structured

In [ ]:
condition_qs = responses.iloc[:, 43:48]
condition_qs.columns

In [ ]:
cs = []
for q in condition_qs.select_dtypes('O').columns:
    for c in condition_qs[q].dropna().unique():
        cs += c.split('|')

Counter(cs).most_common(10)

In [328]:
condition_qs.groupby('Do you currently have a mental health disorder?')['If yes, what condition(s) have you been diagnosed with?'].nunique()

Do you currently have a mental health disorder?
0.0      0
0.5      0
1.0    128
Name: If yes, what condition(s) have you been diagnosed with?, dtype: int64

In [329]:
condition_qs.groupby('Do you currently have a mental health disorder?')['If maybe, what condition(s) do you believe you have?'].nunique()

Do you currently have a mental health disorder?
0.0     0
0.5    99
1.0     0
Name: If maybe, what condition(s) do you believe you have?, dtype: int64

In [332]:
condition_qs.groupby('Have you been diagnosed with a mental health condition by a medical professional?')['If so, what condition(s) were you diagnosed with?'].nunique()

Have you been diagnosed with a mental health condition by a medical professional?
0      0
1    116
Name: If so, what condition(s) were you diagnosed with?, dtype: int64

In [333]:
condition_qs[condition_qs['Do you currently have a mental health disorder?']!=
            condition_qs['Have you been diagnosed with a mental health condition by a medical professional?']].shape

(472, 5)

In [341]:
condition_qs.groupby('Have you been diagnosed with a mental health condition by a medical professional?')['Do you currently have a mental health disorder?'].nunique()

Have you been diagnosed with a mental health condition by a medical professional?
0    3
1    3
Name: Do you currently have a mental health disorder?, dtype: int64

In [334]:
condition_qs[condition_qs['Do you currently have a mental health disorder?']!=
            condition_qs['Have you been diagnosed with a mental health condition by a medical professional?']].sample()

,Do you currently have a mental health disorder?,"If yes, what condition(s) have you been diagnosed with?","If maybe, what condition(s) do you believe you have?",Have you been diagnosed with a mental health condition by a medical professional?,"If so, what condition(s) were you diagnosed with?"
811,0.5,NaN,"Anxiety Disorder (Generalized, Social, Phobia,...",0,NaN


In [318]:
condition_qs[condition_qs['If yes, what condition(s) have you been diagnosed with?']!=
            condition_qs['If so, what condition(s) were you diagnosed with?']].shape

(1064, 3)

In [317]:
condition_qs[condition_qs['If yes, what condition(s) have you been diagnosed with?']!=
            condition_qs['If so, what condition(s) were you diagnosed with?']].sample()

,"If yes, what condition(s) have you been diagnosed with?","If maybe, what condition(s) do you believe you have?","If so, what condition(s) were you diagnosed with?"
1405,NaN,"Personality Disorder (Borderline, Antisocial, ...","Mood Disorder (Depression, Bipolar Disorder, etc)"


In [355]:
condition_qs[condition_qs['If yes, what condition(s) have you been diagnosed with?']!=
            condition_qs['If so, what condition(s) were you diagnosed with?']].sample()

,Do you currently have a mental health disorder?,"If yes, what condition(s) have you been diagnosed with?","If maybe, what condition(s) do you believe you have?",Have you been diagnosed with a mental health condition by a medical professional?,"If so, what condition(s) were you diagnosed with?"
174,1.0,"Anxiety Disorder (Generalized, Social, Phobia,...",NaN,1,"Anxiety Disorder (Generalized, Social, Phobia,..."


In [363]:
ucs = []
for cs in (list(condition_qs['If yes, what condition(s) have you been diagnosed with?'].unique())
               +list(condition_qs['If maybe, what condition(s) do you believe you have?'].unique())
               +list(condition_qs['If so, what condition(s) were you diagnosed with?'].unique())):
    if isinstance(cs, str):
        ucs += cs.split('|')

Counter(ucs).most_common(10)

[('Mood Disorder (Depression, Bipolar Disorder, etc)', 198),
 ('Anxiety Disorder (Generalized, Social, Phobia, etc)', 179),
 ('Attention Deficit Hyperactivity Disorder', 94),
 ('Post-traumatic Stress Disorder', 87),
 ('Personality Disorder (Borderline, Antisocial, Paranoid, etc)', 75),
 ('Obsessive-Compulsive Disorder', 66),
 ('Stress Response Syndromes', 60),
 ('Substance Use Disorder', 60),
 ('Addictive Disorder', 57),
 ('Eating Disorder (Anorexia, Bulimia, etc)', 38)]

In [541]:
current_conditions = pd.DataFrame(index = mhit.index, columns = list(Counter(ucs).keys()), 
                        data = np.array([[0 if (isinstance(mhit.loc[i, 'If yes, what condition(s) have you been diagnosed with?'], float) and 
                                                isinstance(mhit.loc[i, 'If maybe, what condition(s) do you believe you have?'], float))
                                          else 1 if c in str(mhit.loc[i, 'If yes, what condition(s) have you been diagnosed with?'])
                                          else 0.5 if c in str(mhit.loc[i, 'If maybe, what condition(s) do you believe you have?'])
                                          else 0
                                          for c in list(Counter(ucs).keys())] for i in mhit.index]))

In [540]:
diagnosed_conditions = pd.DataFrame(index = mhit.index, columns = list(Counter(ucs).keys()), 
                        data = np.array([[0 if isinstance(mhit.loc[i, 'If so, what condition(s) were you diagnosed with?'], float)  
                                          else 1 if c in mhit.loc[i, 'If so, what condition(s) were you diagnosed with?']
                                          else 0
                                          for c in list(Counter(ucs).keys())] for i in mhit.index]))

In [ ]:
i = np.random.choice(mhit[~mhit['If so, what condition(s) were you diagnosed with?'].isna()].index)

current_conditions.loc[i, current_conditions.columns[current_conditions.loc[i]>0]]

In [394]:
diagnosed_conditions.loc[i, diagnosed_conditions.columns[diagnosed_conditions.loc[i]>0]]

Anxiety Disorder (Generalized, Social, Phobia, etc)    1
Eating Disorder (Anorexia, Bulimia, etc)               1
Name: 467, dtype: int64

In [543]:
diagnosed_conditions.sample()

,"Anxiety Disorder (Generalized, Social, Phobia, etc)","Mood Disorder (Depression, Bipolar Disorder, etc)",Stress Response Syndromes,Substance Use Disorder,Obsessive-Compulsive Disorder,"Eating Disorder (Anorexia, Bulimia, etc)","Personality Disorder (Borderline, Antisocial, Paranoid, etc)",Attention Deficit Hyperactivity Disorder,Addictive Disorder,Post-traumatic Stress Disorder,Pervasive Developmental Disorder (Not Otherwise Specified),Seasonal Affective Disorder,Burn out,PDD-NOS,Dissociative Disorder,Depression,Autism (Asperger's),Traumatic Brain Injury,Gender Dysphoria,Asperges,PTSD (undiagnosed),"Psychotic Disorder (Schizophrenia, Schizoaffective, etc)",Autism,Sexual addiction,"Combination of physical impairment (strongly near-sighted) with a possibly mental one (MCD / ""ADHD"", though its actually a stimulus filtering impairment)",Sleeping Disorder,"I haven't been formally diagnosed, so I felt uncomfortable answering, but Social Anxiety and Depression.",Autism Spectrum Disorder,Transgender,Intimate Disorder,ADD (w/o Hyperactivity),Schizotypal Personality Disorder,Autism spectrum disorder,Suicidal Ideation,"We're all hurt, right?!",Burnout,Gender Identity Disorder,Tinnitus,Depersonalisation,post-partum / anxiety,Asperger Syndrome,Asperger's,depersonalization disorder,PDD-NOS (see above),"Autism - while not a ""mental illness"", still greatly affects how I handle anxiety",Attention Deficit Disorder,"MCD (when it was diagnosed, the ultra-mega ""disorder"" ADHD didn't exist yet)",posttraumatic stress disourder,attention deficit disorder (but not the hyperactive version),Aspergers,autism spectrum disorder,identifier
1123,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,b3f30091-4d87-407f-85e7-d2f562c0b7e8


In [544]:
current_conditions.sample()

,"Anxiety Disorder (Generalized, Social, Phobia, etc)","Mood Disorder (Depression, Bipolar Disorder, etc)",Stress Response Syndromes,Substance Use Disorder,Obsessive-Compulsive Disorder,"Eating Disorder (Anorexia, Bulimia, etc)","Personality Disorder (Borderline, Antisocial, Paranoid, etc)",Attention Deficit Hyperactivity Disorder,Addictive Disorder,Post-traumatic Stress Disorder,Pervasive Developmental Disorder (Not Otherwise Specified),Seasonal Affective Disorder,Burn out,PDD-NOS,Dissociative Disorder,Depression,Autism (Asperger's),Traumatic Brain Injury,Gender Dysphoria,Asperges,PTSD (undiagnosed),"Psychotic Disorder (Schizophrenia, Schizoaffective, etc)",Autism,Sexual addiction,"Combination of physical impairment (strongly near-sighted) with a possibly mental one (MCD / ""ADHD"", though its actually a stimulus filtering impairment)",Sleeping Disorder,"I haven't been formally diagnosed, so I felt uncomfortable answering, but Social Anxiety and Depression.",Autism Spectrum Disorder,Transgender,Intimate Disorder,ADD (w/o Hyperactivity),Schizotypal Personality Disorder,Autism spectrum disorder,Suicidal Ideation,"We're all hurt, right?!",Burnout,Gender Identity Disorder,Tinnitus,Depersonalisation,post-partum / anxiety,Asperger Syndrome,Asperger's,depersonalization disorder,PDD-NOS (see above),"Autism - while not a ""mental illness"", still greatly affects how I handle anxiety",Attention Deficit Disorder,"MCD (when it was diagnosed, the ultra-mega ""disorder"" ADHD didn't exist yet)",posttraumatic stress disourder,attention deficit disorder (but not the hyperactive version),Aspergers,autism spectrum disorder,identifier
563,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5b03c76d-b1c6-496d-88b6-74370b581f6d


In [546]:
all(current_conditions.columns == diagnosed_conditions.columns)

True

In [ ]:
condition_ids = {c: f"C{i+1}" for i, c in enumerate(current_conditions.columns[:-1])}

condition_ids['identifier'] = 'rid'

In [549]:
pd.DataFrame.from_dict({c: [condition_ids[c]] for c in current_conditions}).T.reset_index().rename({'index': 'condition',
                                                                                           0: 'cid'}, axis = 1).head(3)#.to_csv('all_question_ids', index = 0)

,condition,cid
0,"Anxiety Disorder (Generalized, Social, Phobia,...",C1
1,"Mood Disorder (Depression, Bipolar Disorder, etc)",C2
2,Stress Response Syndromes,C3


In [551]:
current_conditions.columns = [condition_ids[c] for c in current_conditions.columns]
current_conditions['rid'] = mhit.index
current_conditions.set_index('rid', inplace = True)
diagnosed_conditions.columns = [condition_ids[c] for c in diagnosed_conditions.columns]
diagnosed_conditions['rid'] = mhit.index
diagnosed_conditions.set_index('rid', inplace = True)

In [552]:
diagnosed_conditions.sample()

,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,C15,C16,C17,C18,C19,C20,C21,C22,C23,C24,C25,C26,C27,C28,C29,C30,C31,C32,C33,C34,C35,C36,C37,C38,C39,C40,C41,C42,C43,C44,C45,C46,C47,C48,C49,C50,C51,rid
1370,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,882c5d5a-f072-4cd8-8797-3b44d930b6cb


In [553]:
current_conditions.sample()

,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,C15,C16,C17,C18,C19,C20,C21,C22,C23,C24,C25,C26,C27,C28,C29,C30,C31,C32,C33,C34,C35,C36,C37,C38,C39,C40,C41,C42,C43,C44,C45,C46,C47,C48,C49,C50,C51,rid
144,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0500eed3-028f-4b6f-80ec-2c012bc8e9ba


In [ ]:
diagnosed_conditions.to_csv('current_conditions_w_ids.csv', index_label = 'rid')
current_conditions.to_csv('diagnosed_conditions_w_ids.csv', index_label = 'rid')

#### 4. positions: unstructured --> structured

In [35]:
demo['Which of the following best describes your work position?'].nunique()

264

In [36]:
demo['Which of the following best describes your work position?'] = demo['Which of the following best describes your work position?'].apply(lambda x: x.replace('|', '/'))

In [37]:
(demo['Which of the following best describes your work position?'].value_counts().cumsum()/demo.shape[0]).head(20)

Which of the following best describes your work position?
Back-end Developer                                             0.183531
Front-end Developer                                            0.270761
Other                                                          0.348918
Supervisor/Team Lead                                           0.396371
Back-end Developer/Front-end Developer                         0.438939
DevOps/SysAdmin                                                0.476622
One-person shop                                                0.511514
Executive Leadership                                           0.543615
Front-end Developer/Back-end Developer                         0.571528
Support                                                        0.595255
Dev Evangelist/Advocate                                        0.614794
Designer                                                       0.634334
Supervisor/Team Lead/Back-end Developer                        0.647592
Front-

In [38]:
positions = []
for p in mhit['Which of the following best describes your work position?'].apply(lambda x: x.replace('|', '/')).unique():
    sep = ['/' if '/' in p else '']
    if sep[0] != '':
        positions+=(p.split(sep[0]))
    else:
        positions.append(p)

unique_positions = list(set(positions))
len(unique_positions)

15

In [39]:
Counter(positions)

Counter({'Back-end Developer': 174,
         'Front-end Developer': 142,
         'Supervisor': 113,
         'Team Lead': 113,
         'DevOps': 104,
         'SysAdmin': 104,
         'Support': 94,
         'One-person shop': 76,
         'Designer': 72,
         'Other': 59,
         'Dev Evangelist': 56,
         'Advocate': 56,
         'Executive Leadership': 50,
         'Sales': 28,
         'HR': 8})

In [40]:
positions = pd.DataFrame(index = demo.index, columns = unique_positions, 
                        data = np.array([[p in mhit.loc[i, 'Which of the following best describes your work position?']
                                          for p in unique_positions] for i in demo.index]))

In [529]:
positions.sample()

,Executive Leadership,Advocate,Supervisor,Front-end Developer,Support,One-person shop,HR,Back-end Developer,Team Lead,Dev Evangelist,Designer,Other,Sales,DevOps,SysAdmin,identifier
1084,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,4a86bf1a-7456-4de3-b684-b89d39da2d8d


In [530]:
position_ids = {c: f"P{i+1}" for i, c in enumerate(positions.columns[:-1])}

In [533]:
pd.DataFrame.from_dict({q: [position_ids[q]] for q in positions}).T.reset_index().rename({'index': 'position',
                                                                                           0: 'pid'}, axis = 1).head(3)#.to_csv('all_question_ids', index = 0)

,position,pid
0,Executive Leadership,P1
1,Advocate,P2
2,Supervisor,P3


In [ ]:
pd.DataFrame.from_dict({q: [position_ids[q]] for q in positions}).T.reset_index().rename({'index': 'position',
                                                                                           0: 'pid'}, axis = 1).to_csv('all_position_ids.csv', 
                                                                                                                       index = 0)
positions.index = mhit.index

positions.columns = [position_ids[c] for c in positions.columns]

In [536]:
positions.sample()

,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,P12,P13,P14,P15,rid
573,False,True,False,True,True,False,False,True,False,True,False,False,False,False,False,2fb57af1-2bec-4f8b-a571-238712cfa708


In [1381]:
positions.to_csv('positions_w_ids.csv', index_label = 'rid')

#### 5. survey responses: qualitative --> quantitative

In [1603]:
responses = mhit.drop(demo.columns, axis = 1)

responses.sample()

,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health concerns and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",Do you think that discussing a mental health disorder with your employer would have negative consequences?,Do you think that discussing a physical health issue with your employer would have negative consequences?,Would you feel comfortable discussing a mental health disorder with your coworkers?,Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?,...,Do you have a family history of mental illness?,Have you had a mental health disorder in the past?,Do you currently have a mental health disorder?,"If yes, what condition(s) have you been diagnosed with?","If maybe, what condition(s) do you believe you have?",Have you been diagnosed with a mental health condition by a medical professional?,"If so, what condition(s) were you diagnosed with?",Have you ever sought treatment for a mental health issue from a mental health professional?,"If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?","If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?"
rid,,,,,,,,,,,,,,,,,,,,,
5e59b3a5-b739-4823-9eef-9555b5ae9859,Yes,No,No,I don't know,Yes,Very difficult,No,No,No,Maybe,...,No,No,No,NaN,NaN,No,NaN,1,Not applicable to me,Not applicable to me


In [199]:
for i, q in enumerate(responses.columns):
    print(f"{i+1}. {q}")

1. Does your employer provide mental health benefits as part of healthcare coverage?
2. Do you know the options for mental health care available under your employer-provided coverage?
3. Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?
4. Does your employer offer resources to learn more about mental health concerns and options for seeking help?
5. Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?
6. If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:
7. Do you think that discussing a mental health disorder with your employer would have negative consequences?
8. Do you think that discussing a physical health issue with your employer would have negative consequences?
9. Would you feel comfortable discussing a mental health disorder with your coworkers?
10. 

In [267]:
Counter(responses.drop(conditions.columns, axis = 1).dtypes)

Counter({dtype('O'): 44, dtype('float64'): 2, dtype('int64'): 2})

In [396]:
responses.drop(conditions.columns, axis = 1).select_dtypes(exclude = 'O').sample(3)

,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health concerns and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",Do you think that discussing a mental health disorder with your employer would have negative consequences?,Do you think that discussing a physical health issue with your employer would have negative consequences?,Would you feel comfortable discussing a mental health disorder with your coworkers?,Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?,Do you feel that your employer takes mental health as seriously as physical health?,Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?,Do you have medical coverage (private insurance or state-provided) which includes treatment of mental health issues?,Do you know local or online resources to seek help for a mental health disorder?,"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?","If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?","If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?","If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?",Do you believe your productivity is ever affected by a mental health issue?,"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",Do you have previous employers?,Have your previous employers provided mental health benefits?,Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,Did your previous employers provide resources to learn more about mental health issues and how to seek help?,Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,Do you think that discussing a mental health disorder with previous employers would have negative consequences?,Do you think that discussing a physical health issue with previous employers would have negative consequences?,Would you have been willing to discuss a mental health issue with your previous co-workers?,Would you have been willing to discuss a mental health issue with your direct supervisor(s)?,Did you feel that your previous employers took mental health as seriously as physical health?,Did you hear of or observe negative consequences for co-workers with mental health issues in your previous workplaces?,Would you be willing to bring up a physical health issue with a potential employer in an interview?,Would you bring up a mental health issue with a potential employer in an interview?,Do you feel that being identified as a person with a mental health issue would hurt your career?,Do you think that team members/co-workers would view you more negatively if they knew you suffered from a mental health issue?,How willing would you be to share with friends and family that you have a mental illness?,Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?,Have your observations of how another individual who discussed a mental health

In [196]:
responses['Does your employer offer resources to learn more about mental health concerns and options for seeking help?'].unique()

array(['No', 'Yes', nan, "I don't know"], dtype=object)

In [264]:
responses["If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:"].unique()

array(['Very easy', 'Somewhat easy', 'Neither easy nor difficult', nan,
       'Very difficult', 'Somewhat difficult', "I don't know"],
      dtype=object)

In [265]:
easy_diff = {'Very easy': 0, 'Somewhat easy': 0.25, 'Neither easy nor difficult': 0.5,
       'Very difficult': 1, 'Somewhat difficult': 0.75, "I don't know": np.nan}

In [266]:
responses['If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:'] = responses['If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:'].map(easy_diff)

In [274]:
responses['How willing would you be to share with friends and family that you have a mental illness?'].unique()

array(['Somewhat open', 'Neutral',
       'Not applicable to me (I do not have a mental illness)',
       'Very open', 'Not open at all', 'Somewhat not open'], dtype=object)

In [277]:
open_not = {'Somewhat open': 0.25, 'Neutral': 0.5,
       'Not applicable to me (I do not have a mental illness)': np.nan,
       'Very open': 0, 'Not open at all': 1, 'Somewhat not open': 0.75}
responses['How willing would you be to share with friends and family that you have a mental illness?'] = responses['How willing would you be to share with friends and family that you have a mental illness?'].map(open_not)

In [268]:
rs = []
for q in [q for q in responses.columns if not q in conditions.columns]:
    if responses[q].dtype == 'O' and responses[q].nunique()<10:
        rs += list(responses[q].unique())

rs_counts = Counter(rs)

rs_counts.most_common(10)

[(nan, 31),
 ('No', 23),
 ('Yes', 21),
 ("I don't know", 12),
 ('Maybe', 11),
 ('Not applicable to me', 6),
 ('Yes, they all did', 4),
 ('Some did', 4),
 ('Yes, always', 3),
 ('None did', 3)]

In [291]:
for q in responses.drop(conditions.columns, axis = 1).columns:
    responses[q] = responses[q].apply(lambda x: 1 if x == 'Yes' or 'Yes,' in str(x) or 'All ' in str(x)
                                      else 0 if x == 'No' or 'No,' in str(x) or 'None' in str(x) or "Never" in str(x) or "Rarely" in str(x) or "Often" in str(x)
                                     else 0.5 if x == 'Maybe' or x == "I don't know" or x == "I'm not sure" or x == "I am not sure" or "Maybe" in str(x)
                                      or 'Some ' in str(x) or 'Sometimes' in str(x)
                                      else np.nan if str(x) == 'Not applicable to me'
                                     else x)

In [299]:
responses.drop(conditions.columns, axis = 1).select_dtypes('O').sample(3)

,Were you aware of the options for mental health care provided by your previous employers?,Why or why not?,Why or why not?.1
701,N/A (not currently aware),A health issue would have a negative effect on...,It would be a negative in the hiring process.
1346,N/A (not currently aware),"If it's going to impede my ability to work, sure","If it has an effect on my ability to do work, ..."
561,N/A (not currently aware),Physical health issues that impact my job are ...,Mental health issues that impact my job are ne...


In [282]:
responses['Does your employer provide mental health benefits as part of healthcare coverage?'] = responses['Does your employer provide mental health benefits as part of healthcare coverage?'].apply(lambda x: 0.5 if isinstance(x, str) else x)

In [282]:
responses['Does your employer provide mental health benefits as part of healthcare coverage?'] = responses['Does your employer provide mental health benefits as part of healthcare coverage?'].apply(lambda x: 0.5 if isinstance(x, str) else x)

In [282]:
responses['Do you know local or online resources to seek help for a mental health disorder?'] = responses['Do you know local or online resources to seek help for a mental health disorder?'].apply(lambda x: 0.5 if isinstance(x, str) else x)

In [297]:
for question in responses.drop(conditions.columns, axis = 1).select_dtypes('O').columns:
    if responses[question].nunique()<10:
        print(responses[question].value_counts())
        for v in responses[question].unique():
            if type(v) == str:
                try:
                    value = float(input(f'replace {v} str with: '))
                    responses[question] = responses[question].apply(lambda x: value if x == v else x)
                except:
                    pass

If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?
1-25%      92
26-50%     72
51-75%     26
76-100%    14
Name: count, dtype: int64


replace 1-25% str with:  0.25
replace 76-100% str with:  1
replace 26-50% str with:  0.5
replace 51-75% str with:  0.75


Were you aware of the options for mental health care provided by your previous employers?
N/A (not currently aware)    582
I was aware of some          384
1                            181
0                            117
Name: count, dtype: int64


replace N/A (not currently aware) str with:  np.nan
replace I was aware of some str with:  0.5


In [288]:
print(responses[question].value_counts())

Do you know local or online resources to seek help for a mental health disorder?
I know some    141
1               83
0               63
Name: count, dtype: int64


In [201]:
for q in responses.columns:
    if responses[q].dtype == 'O':
        try:
            responses[q] = responses[q].map({'No': 0, 'Yes': 1, "I don't know": 0.5, "Maybe": 0.5, "Unsure": 0.5})
        except:
            pass

In [202]:
Counter(responses.dtypes)

Counter({dtype('float64'): 49, dtype('int64'): 2})

In [204]:
mhit['Do you believe your productivity is ever affected by a mental health issue?'].unique()

array([nan, 'Yes', 'Not applicable to me', 'No', 'Unsure'], dtype=object)

In [203]:
responses['Do you believe your productivity is ever affected by a mental health issue?'].unique()

array([nan,  1.,  0.])

In [897]:
responses.to_csv('responses_w_ids.csv', index_label = 'rid')